# JavaScript Prototypes

In JavaScript, a **prototype** is a built-in object from which other objects inherit properties and methods. JavaScript relies entirely on this prototype-based model rather than the classical class-based inheritance found in Java or C++.

---

## The Core Concept: How Prototypes Work

Every object in JavaScript has an internal link to another object, known as its `[[Prototype]]`.

When you access a property or method on an object, JavaScript follows a specific lookup order:

1. It searches the object itself (own properties).
2. If not found, it travels up to the object's prototype.
3. It keeps climbing this **prototype chain** until it finds the property or hits `null`.

```javascript
const animal = {
  eat: true
};

// Create a new object with 'animal' as its prototype
const dog = Object.create(animal);
dog.bark = true;

console.log(dog.bark); // true (found directly on dog)
console.log(dog.eat);  // true (inherited from animal prototype)
```

### The chain always ends at `null`

```
dog  →  animal  →  Object.prototype  →  null
```

```javascript
Object.getPrototypeOf(dog);                     // animal
Object.getPrototypeOf(animal);                  // Object.prototype
Object.getPrototypeOf(Object.prototype);        // null  ← end of the chain
```

This is why every plain object gets `.toString()`, `.hasOwnProperty()`, and `.valueOf()` for free — they live on `Object.prototype`, the last stop before `null`.

---

## `prototype` Property vs `__proto__`

A major point of confusion in JavaScript is the difference between `prototype` and `__proto__`:

- **`prototype`** — a property unique to constructor functions (and classes). It is the blueprint used to assign a prototype to any new instance created with the `new` keyword.
- **`__proto__`** — an accessor property present on all object instances. It points directly to the object's *actual* prototype (the object it inherits from).

```javascript
function User() {}
const u = new User();

u.__proto__ === User.prototype;              // true
Object.getPrototypeOf(u) === User.prototype; // true (preferred form)
u.prototype;                                 // undefined — instances don't have this
```

> ⚠️ **Note:** Using `__proto__` directly in production code is discouraged (it's legacy, kept only for web compatibility). Use [`Object.getPrototypeOf()`](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Reference/Global_Objects/Object/getPrototypeOf) to read and `Object.setPrototypeOf()` to write.

### Avoid `Object.setPrototypeOf()` on existing objects

Mutating an object's prototype *after* creation forces engines to deoptimize that object — V8 throws away its hidden-class optimizations. It's a well-known performance footgun. Prefer `Object.create(proto)` at creation time.

---

## What `new` Actually Does

Understanding `new` demystifies most of this. When you call `new User("Alex")`, the runtime roughly does:

```javascript
function myNew(Constructor, ...args) {
  const obj = Object.create(Constructor.prototype); // 1. link the prototype
  const result = Constructor.apply(obj, args);      // 2. run ctor with `this` = obj
  return (typeof result === 'object' && result !== null) ? result : obj; // 3. return
}
```

That's the whole trick. `new` is prototype linking plus a bound `this`.

---

## Adding Shared Methods via Constructors

Instead of redefining methods every time an object is instantiated, attach them to the constructor's `prototype` object to conserve memory:

```javascript
function User(username) {
  this.username = username; // Unique to each instance
}

// Attached to the prototype — shared across all instances
User.prototype.login = function () {
  console.log(`${this.username} logged in.`);
};

const alex = new User("Alex");
const sam  = new User("Sam");

alex.login(); // "Alex logged in."
alex.login === sam.login; // true — same function object in memory
```

### ⚠️ Don't use arrow functions for prototype methods

Arrow functions have no own `this`; they capture it lexically from the enclosing scope. On a prototype, that scope is the module/global — not the instance.

```javascript
User.prototype.badLogin = () => {
  console.log(this.username); // undefined — `this` is NOT the instance
};
```

Use a regular `function` (or class method shorthand) any time the method needs `this`.

---

## The `constructor` Property

Every `.prototype` object automatically gets a `constructor` property pointing back at the function:

```javascript
User.prototype.constructor === User; // true
alex.constructor === User;           // true (found via the chain)
```

This breaks if you replace the whole prototype object instead of adding to it:

```javascript
User.prototype = {
  login() { /* ... */ }
};

alex2.constructor === User;   // false — now points to Object

// Fix: restore it explicitly
User.prototype = {
  constructor: User,
  login() { /* ... */ }
};
```

---

## Own Properties vs Inherited Properties

Not everything reachable on an object actually *belongs* to it. This distinction matters constantly in real code.

```javascript
dog.hasOwnProperty('bark'); // true  — own property
dog.hasOwnProperty('eat');  // false — inherited from animal
'eat' in dog;               // true  — `in` walks the whole chain
```

| Technique | Walks prototype chain? |
|---|---|
| `for...in` | ✅ Yes (enumerable inherited props included) |
| `Object.keys()` / `Object.values()` / `Object.entries()` | ❌ No — own enumerable only |
| `in` operator | ✅ Yes |
| `Object.hasOwn(obj, key)` *(ES2022, preferred)* | ❌ No |
| `obj.hasOwnProperty(key)` | ❌ No |

Prefer `Object.hasOwn(obj, key)` over `obj.hasOwnProperty(key)` — it still works when the object has no prototype or has shadowed `hasOwnProperty`.

### Shadowing

Assigning to an inherited property name creates an **own** property that hides the prototype's version; it never modifies the prototype.

```javascript
dog.eat = false;
console.log(dog.eat);    // false (own)
console.log(animal.eat); // true  (untouched)

delete dog.eat;
console.log(dog.eat);    // true  — the inherited one resurfaces
```

---

## Inheritance Between Constructors

Before `class`, chaining two constructors took two explicit steps:

```javascript
function Animal(name) {
  this.name = name;
}
Animal.prototype.speak = function () {
  console.log(`${this.name} makes a sound.`);
};

function Dog(name, breed) {
  Animal.call(this, name);   // 1. inherit instance properties
  this.breed = breed;
}

// 2. inherit prototype methods
Dog.prototype = Object.create(Animal.prototype);
Dog.prototype.constructor = Dog;

Dog.prototype.speak = function () {
  console.log(`${this.name} barks.`);
};
```

Chain: `instance → Dog.prototype → Animal.prototype → Object.prototype → null`

---

## Classes are Just "Syntactic Sugar"

Modern JavaScript uses the `class` keyword. Under the hood it's a cleaner interface sitting on top of the same prototype system.

```javascript
class Vehicle {
  constructor(type) {
    this.type = type;
  }

  move() {                      // → Vehicle.prototype.move
    console.log("Moving...");
  }

  static compare(a, b) {        // → Vehicle.compare (on the function itself)
    return a.type === b.type;
  }
}

class Car extends Vehicle {
  constructor(type, doors) {
    super(type);                // ≡ Vehicle.call(this, type)
    this.doors = doors;
  }
}
```

`move()` is automatically assigned to `Vehicle.prototype` by the runtime. `extends` wires `Car.prototype`'s `[[Prototype]]` to `Vehicle.prototype` for you, and `super()` replaces the manual `Parent.call(this, ...)`.

**Static methods** land on the constructor function itself, not on `.prototype`, so instances can't see them:

```javascript
const c = new Car("sedan", 4);
c.compare;        // undefined
Vehicle.compare;  // function
```

### Where classes are *not* just sugar

- Class bodies always run in strict mode.
- Class declarations are not hoisted in a usable way (temporal dead zone).
- Calling a class without `new` throws a `TypeError`; a constructor function silently misbehaves.
- `#privateField` syntax has no prototype-based equivalent.

---

## Why Prototypes Matter

- **Memory optimization** — sharing methods across millions of instances via a single prototype reference prevents duplication and saves RAM.
- **Dynamic upgrades** — add a method to a prototype at runtime and every existing *and* future instance instantly gains access to it. This is live, not copied.
- **Built-in functions** — native features like `Array#map()` and `String#toLowerCase()` are powered by `Array.prototype` and `String.prototype`.

```javascript
function Cat(n) { this.n = n; }
const kitty = new Cat("Kitty");

Cat.prototype.meow = function () { console.log("meow"); };
kitty.meow(); // works, even though kitty existed first
```

---

## Gotchas & Practical Notes

### 1. Don't extend built-in prototypes

```javascript
Array.prototype.last = function () { return this[this.length - 1]; }; // ❌
```

It looks handy but it's globally visible, pollutes `for...in` over arrays, and collides with future language additions. (The real-world cautionary tale: `Array.prototype.flatten` from MooTools broke a proposed TC39 method, which had to be renamed to `flat`.)

### 2. Never share mutable state on a prototype

```javascript
function Team() {}
Team.prototype.members = [];   // ❌ one array shared by ALL instances

const a = new Team(), b = new Team();
a.members.push("Alex");
console.log(b.members);        // ["Alex"] — surprise
```

Put mutable state on the instance inside the constructor. Prototypes are for behavior (methods), not data.

### 3. `instanceof` walks the prototype chain

```javascript
kitty instanceof Cat;    // true
kitty instanceof Object; // true
```

It checks whether `Cat.prototype` appears anywhere in `kitty`'s chain — it does not compare constructors. Reassigning `Cat.prototype` afterwards changes the answer for old instances.

### 4. `Object.create(null)` for pure dictionaries

```javascript
const map = Object.create(null);
map.toString;                  // undefined — no inherited junk
```

Useful when using an object as a lookup table where keys are user input, so a key named `constructor` or `toString` can't collide with inherited members.

### 5. Prototype pollution (security)

If an attacker can write to `Object.prototype` — typically via an unsafe recursive merge or `JSON.parse` of untrusted input with a `__proto__` key — every object in the application inherits their property. This can escalate to authentication bypass or RCE on Node.

```javascript
// vulnerable merge
merge({}, JSON.parse('{"__proto__": {"isAdmin": true}}'));
({}).isAdmin; // true — every object is now admin
```

Mitigations: validate keys against `__proto__` / `constructor` / `prototype`, use `Object.create(null)` for maps, use `Map` instead of objects for dynamic keys, and `Object.freeze(Object.prototype)` in hardened environments.

---

## Quick Reference

| Task | API |
|---|---|
| Read an object's prototype | `Object.getPrototypeOf(obj)` |
| Create an object with a given prototype | `Object.create(proto)` |
| Set a prototype (avoid post-creation) | `Object.setPrototypeOf(obj, proto)` |
| Check own property | `Object.hasOwn(obj, key)` |
| Check anywhere in the chain | `key in obj` |
| Check chain membership by constructor | `obj instanceof Ctor` |
| Prototype-free object | `Object.create(null)` |

---

## References

- [GeeksforGeeks — JS Prototype](https://www.geeksforgeeks.org/javascript/js-prototype/)
- [Namaste Dev — JavaScript Prototypes Explained](https://namastedev.com/blog/javascript-prototypes-explained/)
- [freeCodeCamp — How `__proto__`, `prototype` and inheritance actually work](https://www.freecodecamp.org/news/how-proto-prototype-and-inheritance-actually-work-in-javascript/)
- [MDN — `Object.getPrototypeOf()`](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Reference/Global_Objects/Object/getPrototypeOf)
- [Stack Overflow — How does JavaScript prototype work?](https://stackoverflow.com/questions/572897/how-does-javascript-prototype-work)
- [PortSwigger — JavaScript prototypes and inheritance / prototype pollution](https://portswigger.net/web-security/prototype-pollution/javascript-prototypes-and-inheritance)
- [Mimo — Prototype glossary](https://mimo.org/glossary/javascript/prototype)